<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My answer
For Lane 2 — Refresh / Content Opportunity Scoring, one row represents the performance of one content item for one client on one report date.

I use the fact_content_daily_performance table as the main source. I use March 2026 as the development window because it is a mid-panel month rather than the final _sample month.

The analysis is intended to support prioritizing content pages for human review for possible refresh, improvement, protection, pruning, or monitoring.




In [39]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Table: fact_content_daily_performance")
print("Target development month: 2026-03")
print("Expected grain: one row per report date, client, and content item")

Lane: Refresh / Content Opportunity Scoring
Table: fact_content_daily_performance
Target development month: 2026-03
Expected grain: one row per report date, client, and content item


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features

1.gsc_impressions — knowable at the decision moment because it records observed search exposure for the content item.
2.gsc_clicks — knowable at the decision moment because it records observed search clicks.
3.gsc_avg_position — knowable at the decision moment because it summarizes observed search position.
4.ga4_engaged_sessions — knowable at the decision moment because it records observed engaged sessions.
5.ga4_total_engagement_sec — knowable at the decision moment because it records observed engagement time.


Label / proxy

The label will be a future page-performance outcome used as a proxy for whether a content item deserves attention. It must be calculated from a later time period so that the features represent information available at the decision moment.

Context

report_date — identifies when the observation was recorded.
month — identifies the warehouse month.
client_hash_id — identifies the anonymized client.
content_hash_id — identifies the anonymized content item.
gsc_data_available — indicates whether GSC data is available.
ga4_data_available — indicates whether GA4 data is available.

Excluded

I will exclude fields that directly contain or are derived from the future outcome being predicted. I will also exclude private or identifying information such as URLs, client names, and private queries. A field will only be used as a feature if it would have been available at the decision moment.

In [40]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [41]:
query1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_page_day_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

print(con.sql(query1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ unique_page_day_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │              9841378 │
└────────────┴──────────────────────┘



In [42]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

print(con.sql(query2))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [43]:
query3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

print(con.sql(query3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



In [44]:
features_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_engaged_sessions,
    ga4_total_engagement_sec
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
LIMIT 10
"""

feature_df = con.sql(features_query).df()

print("Rows shown:", len(feature_df))
print("Feature count:", 5)
display(feature_df)

Rows shown: 10
Feature count: 5


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,ga4_total_engagement_sec
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,0,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,0,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,0,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,0,0
5,2026-03-01,client_65de48885f4ef01b,content_872342e050545a12,39,0,6.538462,0,0
6,2026-03-01,client_65de48885f4ef01b,content_3c286ded8bd68120,88,1,8.431818,0,0
7,2026-03-01,client_65de48885f4ef01b,content_b2108e8fe3360fa6,40,1,5.300000,0,0
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,0,30.304348,0,0
9,2026-03-01,client_65de48885f4ef01b,content_bd07be40ea0d5f54,23,0,5.478261,0,0


In [45]:
availability_rate = 3611061 / 9841378
print("GSC availability rate:", round(availability_rate, 3))

GSC availability rate: 0.367


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations that affect what I can conclude.

The data is observational, so it can show relationships and patterns but cannot prove that changing a page will cause better search performance.
GSC and GA4 data are not available for every row, so some content items have incomplete signals.
The analysis uses March 2026 as a development window. A single month may not represent longer-term page behavior.
The final _sample month should be treated as a sealed test period rather than used to develop the label or features.
Client and content identifiers are anonymized, so I cannot use real client names, URLs, or private search queries.
A future outcome label must not be used as a feature because that would create data leakage.



In [46]:
print("Data limits checked:")
print("- Observational data: no causal claims")
print("- GSC/GA4 availability varies")
print("- Development window: March 2026")
print("- Final month reserved as a sealed test period")
print("- Identifiers are anonymized")
print("- Future outcome fields excluded from features")

Data limits checked:
- Observational data: no causal claims
- GSC/GA4 availability varies
- Development window: March 2026
- Final month reserved as a sealed test period
- Identifiers are anonymized
- Future outcome fields excluded from features


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.